# BikeStores DWH — 01 Staging & Data Cleansing

Notebook ini digunakan untuk tahap awal pipeline:

```text
RAW CSV
   │
   ▼
Load & Validation
   │
   ▼
Staging Raw
   │
   ▼
Data Cleansing
   │
   ▼
Staging Clean
   │
   ▼
Post-Cleansing Validation
```

Notebook ini **belum membangun dimensi atau fact table**.

Tujuan tahap ini adalah memastikan source mentah:
- berhasil dibaca;
- tidak dimodifikasi;
- divalidasi;
- dibersihkan sesuai rule yang sudah didokumentasikan;
- menghasilkan evidence before/after dan cleansing log.


## 00 — Setup Environment

### Struktur folder yang digunakan

```text
BikeStores_DWH/
├── data/
│   ├── raw/
│   ├── staging/
│   │   ├── raw/
│   │   └── clean/
│   └── evidence/
│       └── cleansing/
└── logs/
```

> Untuk Google Colab, direkomendasikan menyimpan project di Google Drive agar file tidak hilang ketika runtime berhenti.


In [ ]:
# Jalankan cell ini di Google Colab.
# Jika dijalankan di luar Colab, bagian mount otomatis dilewati.

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import json
import os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/BikeStores_DWH')
else:
    # Ubah bila dijalankan di komputer lokal.
    PROJECT_DIR = Path.cwd() / 'BikeStores_DWH'

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
STAGING_RAW_DIR = PROJECT_DIR / 'data' / 'staging' / 'raw'
STAGING_CLEAN_DIR = PROJECT_DIR / 'data' / 'staging' / 'clean'
EVIDENCE_DIR = PROJECT_DIR / 'data' / 'evidence' / 'cleansing'
LOG_DIR = PROJECT_DIR / 'logs'

for directory in [
    RAW_DIR,
    STAGING_RAW_DIR,
    STAGING_CLEAN_DIR,
    EVIDENCE_DIR,
    LOG_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR       :', PROJECT_DIR)
print('RAW_DIR           :', RAW_DIR)
print('STAGING_RAW_DIR   :', STAGING_RAW_DIR)
print('STAGING_CLEAN_DIR :', STAGING_CLEAN_DIR)
print('EVIDENCE_DIR      :', EVIDENCE_DIR)
print('LOG_DIR           :', LOG_DIR)


## 01 — Menyiapkan Raw Dataset

Notebook menggunakan strategi **local-first, Kaggle fallback**.

Alurnya:

```text
Cek data/raw/
    │
    ├── semua file ada
    │      └── lanjut tanpa download
    │
    └── ada file yang hilang
           └── download dataset dari Kaggle
                  │
                  ├── cari CSV secara rekursif
                  ├── copy hanya file yang belum ada
                  └── validasi ulang 9 file
```

Dataset Kaggle:

```text
dillonmyrick/bike-store-sample-database
```

Folder `data/raw/` tetap dianggap **immutable source**. File yang sudah ada tidak ditimpa.


In [ ]:
EXPECTED_FILES = [
    'brands.csv',
    'categories.csv',
    'customers.csv',
    'order_items.csv',
    'orders.csv',
    'products.csv',
    'staffs.csv',
    'stocks.csv',
    'stores.csv',
]

RAW_DIR.mkdir(parents=True, exist_ok=True)

missing_files = [
    filename
    for filename in EXPECTED_FILES
    if not (RAW_DIR / filename).exists()
]

if missing_files:
    print('Dataset belum lengkap.')
    print('File yang belum ditemukan:')
    for filename in missing_files:
        print(' -', filename)

    print('\nMencoba download dari Kaggle...')

    # Install hanya bila kagglehub belum tersedia.
    try:
        import kagglehub
    except ImportError:
        import sys
        import subprocess

        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            'kagglehub'
        ])

        import kagglehub

    kaggle_path = Path(
        kagglehub.dataset_download(
            'dillonmyrick/bike-store-sample-database'
        )
    )

    print('Dataset Kaggle tersimpan sementara di:')
    print(kaggle_path)

    # Cari seluruh CSV, termasuk bila tersimpan dalam subfolder.
    downloaded_csv = {}

    for csv_file in kaggle_path.rglob('*.csv'):
        downloaded_csv[csv_file.name] = csv_file

    for filename in EXPECTED_FILES:
        destination = RAW_DIR / filename

        # Hanya copy file yang belum tersedia.
        if destination.exists():
            continue

        source = downloaded_csv.get(filename)

        if source is not None:
            shutil.copy2(source, destination)
            print(f'Copied: {filename}')
        else:
            print(f'Tidak ditemukan di hasil download: {filename}')

else:
    print('Semua dataset sudah tersedia.')
    print('Download Kaggle dilewati.')

# Validasi ulang setelah fallback Kaggle.
missing_after_download = [
    filename
    for filename in EXPECTED_FILES
    if not (RAW_DIR / filename).exists()
]

if missing_after_download:
    raise FileNotFoundError(
        'Dataset masih belum lengkap: '
        + ', '.join(missing_after_download)
    )

print('\nPASS: seluruh 9 file dataset tersedia.')


### Salin Raw → Staging Raw

Tujuannya adalah menjaga source asli tetap immutable.

```text
data/raw/
   │
   │ copy
   ▼
data/staging/raw/
```

Semua proses selanjutnya menggunakan staging copy.


In [ ]:
for filename in EXPECTED_FILES:
    src = RAW_DIR / filename
    dst = STAGING_RAW_DIR / filename
    shutil.copy2(src, dst)

print(f'{len(EXPECTED_FILES)} file disalin ke staging/raw.')


## 02 — Load dan Pre-Cleansing Validation

Pada tahap ini kita memuat seluruh CSV dari `staging/raw` dan melakukan validasi awal.

Validasi utama:
- row count;
- duplicate penuh;
- missing value;
- candidate key;
- foreign key;
- domain numeric;
- parse tanggal.


In [ ]:
dfs = {
    filename: pd.read_csv(STAGING_RAW_DIR / filename)
    for filename in EXPECTED_FILES
}

for filename, df in dfs.items():
    print(
        f'{filename:<18} '
        f'rows={len(df):>5} | columns={len(df.columns):>2}'
    )


In [ ]:
# Ringkasan kondisi sebelum cleansing
before_summary = []

for filename, df in dfs.items():
    before_summary.append({
        'file': filename,
        'rows': len(df),
        'columns': len(df.columns),
        'full_row_duplicates': int(df.duplicated().sum()),
        'total_null_values': int(df.isna().sum().sum()),
    })

before_summary = pd.DataFrame(before_summary)
before_summary


### Validasi Candidate Key

Candidate key yang digunakan untuk pemeriksaan:

```text
brands       : brand_id
categories   : category_id
customers    : customer_id
orders       : order_id
order_items  : order_id + item_id
products     : product_id
staffs       : staff_id
stocks       : store_id + product_id
stores       : store_id
```


In [ ]:
PK_SPECS = {
    'brands.csv': ['brand_id'],
    'categories.csv': ['category_id'],
    'customers.csv': ['customer_id'],
    'orders.csv': ['order_id'],
    'order_items.csv': ['order_id', 'item_id'],
    'products.csv': ['product_id'],
    'staffs.csv': ['staff_id'],
    'stocks.csv': ['store_id', 'product_id'],
    'stores.csv': ['store_id'],
}

pk_checks = []

for filename, key_cols in PK_SPECS.items():
    df = dfs[filename]

    duplicate_rows = int(df.duplicated(subset=key_cols).sum())
    null_rows = int(df[key_cols].isna().any(axis=1).sum())

    pk_checks.append({
        'file': filename,
        'candidate_key': ' + '.join(key_cols),
        'duplicate_key_rows': duplicate_rows,
        'null_in_key': null_rows,
        'status': 'PASS' if duplicate_rows == 0 and null_rows == 0 else 'REVIEW'
    })

pk_checks = pd.DataFrame(pk_checks)
pk_checks


### Validasi Foreign Key

Relasi utama OLTP diperiksa agar record orphan tidak masuk ke pipeline tanpa diketahui.


In [ ]:
FK_SPECS = [
    ('products.csv', 'brand_id', 'brands.csv', 'brand_id'),
    ('products.csv', 'category_id', 'categories.csv', 'category_id'),
    ('orders.csv', 'customer_id', 'customers.csv', 'customer_id'),
    ('orders.csv', 'store_id', 'stores.csv', 'store_id'),
    ('orders.csv', 'staff_id', 'staffs.csv', 'staff_id'),
    ('order_items.csv', 'order_id', 'orders.csv', 'order_id'),
    ('order_items.csv', 'product_id', 'products.csv', 'product_id'),
    ('staffs.csv', 'store_id', 'stores.csv', 'store_id'),
    ('staffs.csv', 'manager_id', 'staffs.csv', 'staff_id'),
    ('stocks.csv', 'store_id', 'stores.csv', 'store_id'),
    ('stocks.csv', 'product_id', 'products.csv', 'product_id'),
]

fk_checks = []

for child_file, child_col, parent_file, parent_col in FK_SPECS:
    child = dfs[child_file]
    parent_values = set(dfs[parent_file][parent_col].dropna())

    invalid_mask = (
        child[child_col].notna()
        & ~child[child_col].isin(parent_values)
    )

    orphan_rows = int(invalid_mask.sum())

    fk_checks.append({
        'child': f'{child_file}.{child_col}',
        'parent': f'{parent_file}.{parent_col}',
        'orphan_rows': orphan_rows,
        'status': 'PASS' if orphan_rows == 0 else 'REVIEW'
    })

fk_checks = pd.DataFrame(fk_checks)
fk_checks


### Validasi Business Rule Dasar

Rule yang diperiksa sebelum cleansing:

```text
quantity > 0
discount berada pada 0..1
list_price >= 0
stock quantity >= 0
required_date >= order_date
shipped_date >= order_date
Completed harus mempunyai shipped_date
```

Late shipment (`shipped_date > required_date`) **bukan error**.


In [ ]:
orders = dfs['orders.csv'].copy()

for col in ['order_date', 'required_date', 'shipped_date']:
    orders[col + '_parsed'] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

business_checks = pd.DataFrame([
    {
        'check': 'required_date < order_date',
        'count': int(
            (orders['required_date_parsed'] < orders['order_date_parsed']).sum()
        )
    },
    {
        'check': 'shipped_date < order_date',
        'count': int(
            (orders['shipped_date_parsed'] < orders['order_date_parsed']).sum()
        )
    },
    {
        'check': 'Completed tanpa shipped_date',
        'count': int(
            ((orders['order_status'] == 4) & orders['shipped_date'].isna()).sum()
        )
    },
    {
        'check': 'Non-completed dengan shipped_date',
        'count': int(
            ((orders['order_status'] != 4) & orders['shipped_date'].notna()).sum()
        )
    },
    {
        'check': 'quantity <= 0',
        'count': int(
            (dfs['order_items.csv']['quantity'] <= 0).sum()
        )
    },
    {
        'check': 'discount < 0 atau > 1',
        'count': int(
            (
                (dfs['order_items.csv']['discount'] < 0)
                | (dfs['order_items.csv']['discount'] > 1)
            ).sum()
        )
    },
    {
        'check': 'list_price < 0 pada order_items',
        'count': int(
            (dfs['order_items.csv']['list_price'] < 0).sum()
        )
    },
    {
        'check': 'list_price < 0 pada products',
        'count': int(
            (dfs['products.csv']['list_price'] < 0).sum()
        )
    },
    {
        'check': 'stock quantity < 0',
        'count': int(
            (dfs['stocks.csv']['quantity'] < 0).sum()
        )
    },
])

business_checks


## 03 — Data Cleansing

### Rule yang diterapkan

Pada tahap ini kita hanya melakukan perubahan yang sudah mempunyai dasar dari profiling dan Data Quality Checklist.

```text
RULE 01
Trim leading/trailing whitespace pada seluruh kolom string.

RULE 02
Parse order_date, required_date, shipped_date sebagai tanggal.

RULE 03
Pertahankan NULL yang valid.

RULE 04
Jangan menghapus late shipment.

RULE 05
Jangan melakukan deduplication bila tidak ada duplicate.

RULE 06
Jangan mengubah source raw.
```

Cleansing dilakukan pada copy dataframe.


In [ ]:
clean_dfs = {
    filename: df.copy(deep=True)
    for filename, df in dfs.items()
}

cleansing_log = []


### 03.1 Trim Whitespace

Kita menggunakan `strip()` hanya untuk leading/trailing whitespace.

Contoh:

```text
"123 Main St   "
        ↓
"123 Main St"
```

Isi teks di tengah string tidak diubah.


In [ ]:
for filename, df in clean_dfs.items():
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].notna()

        before = df.loc[non_null, col].astype(str)
        after = before.str.strip()

        affected = int((before != after).sum())

        # Hanya replace nilai non-null; NULL tetap NULL
        df.loc[non_null, col] = after

        if affected > 0:
            cleansing_log.append({
                'file': filename,
                'column': col,
                'rule': 'TRIM_WHITESPACE',
                'rows_affected': affected,
                'action': 'CLEAN'
            })

print('Whitespace cleansing selesai.')


### 03.2 Parse Tanggal

Tanggal di `orders` dikonversi menjadi tipe datetime pada staging memory.

`shipped_date` yang kosong **tetap NaT/NULL**.


In [ ]:
for col in ['order_date', 'required_date', 'shipped_date']:
    source_non_null = clean_dfs['orders.csv'][col].notna()

    parsed = pd.to_datetime(
        clean_dfs['orders.csv'][col],
        errors='coerce'
    )

    parse_failures = int(
        (source_non_null & parsed.isna()).sum()
    )

    if parse_failures > 0:
        raise ValueError(
            f'Ditemukan {parse_failures} kegagalan parse pada {col}'
        )

    clean_dfs['orders.csv'][col] = parsed

    cleansing_log.append({
        'file': 'orders.csv',
        'column': col,
        'rule': 'PARSE_DATE',
        'rows_affected': int(source_non_null.sum()),
        'action': 'STANDARDIZE_TYPE'
    })

print('Date parsing selesai.')


### 03.3 Pertahankan NULL yang Valid

Tidak ada `fillna()` otomatis.

Secara khusus:

```text
customers.phone NULL
→ dipertahankan

orders.shipped_date NULL pada non-completed order
→ dipertahankan

staffs.manager_id NULL
→ dipertahankan
```

Cell berikut hanya memverifikasi bahwa cleansing tidak mengubah jumlah NULL pada field-field tersebut.


In [ ]:
null_preservation_checks = pd.DataFrame([
    {
        'field': 'customers.phone',
        'before_null': int(dfs['customers.csv']['phone'].isna().sum()),
        'after_null': int(clean_dfs['customers.csv']['phone'].isna().sum()),
    },
    {
        'field': 'orders.shipped_date',
        'before_null': int(dfs['orders.csv']['shipped_date'].isna().sum()),
        'after_null': int(clean_dfs['orders.csv']['shipped_date'].isna().sum()),
    },
    {
        'field': 'staffs.manager_id',
        'before_null': int(dfs['staffs.csv']['manager_id'].isna().sum()),
        'after_null': int(clean_dfs['staffs.csv']['manager_id'].isna().sum()),
    },
])

null_preservation_checks['status'] = np.where(
    null_preservation_checks['before_null']
    == null_preservation_checks['after_null'],
    'PASS',
    'REVIEW'
)

null_preservation_checks


## 04 — Post-Cleansing Validation

Setelah cleansing, kita memastikan:
- row count tidak berubah;
- candidate key tetap unik;
- foreign key tetap valid;
- whitespace luar sudah hilang;
- NULL valid tetap dipertahankan.


In [ ]:
after_summary = []

for filename, df in clean_dfs.items():
    after_summary.append({
        'file': filename,
        'rows': len(df),
        'columns': len(df.columns),
        'full_row_duplicates': int(df.duplicated().sum()),
        'total_null_values': int(df.isna().sum().sum()),
    })

after_summary = pd.DataFrame(after_summary)

row_count_check = before_summary[['file', 'rows']].merge(
    after_summary[['file', 'rows']],
    on='file',
    suffixes=('_before', '_after')
)

row_count_check['status'] = np.where(
    row_count_check['rows_before'] == row_count_check['rows_after'],
    'PASS',
    'REVIEW'
)

row_count_check


In [ ]:
# Pastikan tidak ada lagi leading/trailing whitespace
whitespace_after = []

for filename, df in clean_dfs.items():
    for col in df.select_dtypes(include='object').columns:
        s = df[col].dropna().astype(str)
        count = int((s != s.str.strip()).sum())

        if count > 0:
            whitespace_after.append({
                'file': filename,
                'column': col,
                'remaining_whitespace_rows': count
            })

whitespace_after = pd.DataFrame(whitespace_after)

if whitespace_after.empty:
    print('PASS: tidak ada leading/trailing whitespace tersisa.')
else:
    display(whitespace_after)


## 05 — Simpan Staging Clean dan Evidence

CSV output tetap dipisahkan dari raw source.

Untuk kompatibilitas CSV:
- kolom datetime ditulis sebagai `YYYY-MM-DD`;
- NULL tetap kosong;
- source raw tidak ditimpa.


In [ ]:
# Simpan staging clean
for filename, df in clean_dfs.items():
    output_df = df.copy()

    if filename == 'orders.csv':
        for col in ['order_date', 'required_date', 'shipped_date']:
            output_df[col] = output_df[col].dt.strftime('%Y-%m-%d')

    output_df.to_csv(
        STAGING_CLEAN_DIR / filename,
        index=False
    )

# Evidence
cleansing_log_df = pd.DataFrame(cleansing_log)

before_summary.to_csv(
    EVIDENCE_DIR / 'before_summary.csv',
    index=False
)

after_summary.to_csv(
    EVIDENCE_DIR / 'after_summary.csv',
    index=False
)

cleansing_log_df.to_csv(
    EVIDENCE_DIR / 'cleansing_log.csv',
    index=False
)

row_count_check.to_csv(
    EVIDENCE_DIR / 'row_count_validation.csv',
    index=False
)

null_preservation_checks.to_csv(
    EVIDENCE_DIR / 'null_preservation_validation.csv',
    index=False
)

pk_checks.to_csv(
    EVIDENCE_DIR / 'preclean_pk_checks.csv',
    index=False
)

fk_checks.to_csv(
    EVIDENCE_DIR / 'preclean_fk_checks.csv',
    index=False
)

business_checks.to_csv(
    EVIDENCE_DIR / 'preclean_business_checks.csv',
    index=False
)

print('Staging clean tersimpan di :', STAGING_CLEAN_DIR)
print('Evidence tersimpan di      :', EVIDENCE_DIR)


## 06 — Expected Result

Dengan dataset BikeStores yang digunakan pada proyek ini, hasil penting yang diharapkan antara lain:

```text
customers.street
before  : 1,445 record memiliki outer whitespace
after   : 0

Full-row duplicate
before  : 0
after   : 0

Orphan FK
before  : 0
after   : 0

orders.shipped_date NULL
before  : 170
after   : 170
```

Jumlah row setiap tabel **harus tetap sama** karena pada cleansing tahap ini tidak ada record yang perlu dibuang.

Jika hasil berbeda, hentikan proses dan periksa sebelum masuk ke ETL dimensional.
